# Лабораторна робота № 4. ETL-конвеєр для обробки даних

Предметна область – роздрібна торгівля (проєкт «Retail Analytics»), набір даних Online Retail II
(UCI Machine Learning Repository). Конвеєр завантажує сирі продажі (Bronze), очищує їх (Silver),
будує виміри (Gold) і виконує вісім аналітичних завдань на PySpark та Spark SQL.

## 1. Налаштування середовища

Перед запуском у Google Colab файли `online_retail_2009-2010.csv` і `online_retail_2010-2011.csv`
потрібно завантажити на Google Drive у папку `MyDrive/retail-data/`.

In [ ]:
!pip install pyspark

In [ ]:
import sys

from pyspark import SparkConf
from pyspark.sql import SparkSession, Window
import pyspark.sql.types as T
import pyspark.sql.functions as F

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/retail-data/'
else:
    DATA_DIR = 'data/'
RESULTS_DIR = DATA_DIR + 'retail-results/'

# master('local[*]') - використовує всі доступні ядра для обчислень
spark_session = (SparkSession.builder
                 .master('local[*]')
                 .appName('retail etl')
                 .config(conf=SparkConf())
                 .getOrCreate())
print('Spark', spark_session.version)

## 2. Допоміжні функції ETL: E – Extract і L – Load

In [ ]:
def read_data(spark_session, file_name: str, schema: T.StructType):
    """Читає CSV-файл Online Retail II із заданою схемою."""
    return spark_session.read.csv(DATA_DIR + file_name,
                                  header=True,
                                  schema=schema,
                                  timestampFormat='yyyy-MM-dd HH:mm:ss',
                                  escape='"',          # лапки в назвах товарів записано як ""
                                  mode='FAILFAST')     # помилка, якщо рядок не відповідає схемі


def write_data(df, task_name: str):
    """Записує результат завдання в один CSV-файл у каталозі результатів."""
    output_dir = RESULTS_DIR + task_name
    print(f'Запис результату у: {output_dir}')
    df.coalesce(1).write.csv(output_dir, header=True, mode='overwrite')


def write_layer(df, layer_name: str):
    """Записує шар сховища (Silver або Gold) у форматі Parquet."""
    output_dir = DATA_DIR + 'lakehouse/' + layer_name
    print(f'Запис шару у: {output_dir}')
    df.write.parquet(output_dir, mode='overwrite')

In [ ]:
# Тестовий DataFrame: три рядки чеків
test_df = spark_session.createDataFrame(
    [('536365', '85123A', 'WHITE HANGING HEART T-LIGHT HOLDER', 6, 2.55),
     ('536365', '71053', 'WHITE METAL LANTERN', 6, 3.39),
     ('C536379', 'D', 'Discount', -1, 27.50)],
    schema='invoice string, stock_code string, description string, quantity int, price double')

test_df.withColumn('revenue', F.round(F.col('quantity') * F.col('price'), 2)).show(truncate=False)

## 3. Extract: завантаження сирих даних (Bronze)

Набір складається з двох файлів: продажі за 01.12.2009–09.12.2010 і за 01.12.2010–09.12.2011.

In [ ]:
schema_sales = T.StructType([
    T.StructField('Invoice', T.StringType(), False),       # номер чека, 'C...' - скасування
    T.StructField('StockCode', T.StringType(), False),     # код товару
    T.StructField('Description', T.StringType(), True),
    T.StructField('Quantity', T.IntegerType(), False),
    T.StructField('InvoiceDate', T.TimestampType(), False),
    T.StructField('Price', T.DoubleType(), False),         # ціна за одиницю, £
    T.StructField('Customer ID', T.DoubleType(), True),    # порожній для покупок без облікового запису
    T.StructField('Country', T.StringType(), True),
])

bronze_df = (read_data(spark_session, 'online_retail_2009-2010.csv', schema_sales)
             .unionByName(read_data(spark_session, 'online_retail_2010-2011.csv', schema_sales)))

bronze_df.printSchema()
print('Рядків у сирих даних:', bronze_df.count())
bronze_df.show(5, truncate=False)

## 4. Transform: очищення даних (Silver)

Правила очищення: прибрати повні дублікати (перетин файлів за 1–9 грудня 2010 р.), привести назви й типи
стовпців, позначити покупки без облікового запису кодом 0, відокремити скасування та службові коди
(поштові витрати, знижки, корекції).

In [ ]:
typed_df = (bronze_df
    .dropDuplicates()
    .select(F.col('Invoice').alias('invoice'),
            F.col('StockCode').alias('stock_code'),
            F.trim('Description').alias('description'),
            F.col('Quantity').alias('quantity'),
            F.col('InvoiceDate').alias('invoice_date'),
            F.col('Price').alias('unit_price'),
            F.coalesce(F.col('Customer ID').cast('int'), F.lit(0)).alias('customer_id'),  # 0 - гість
            F.col('Country').alias('country'))
    .withColumn('is_cancelled', F.col('invoice').startswith('C'))
    .withColumn('is_product', F.col('stock_code').rlike(r'^\d{5}[A-Za-z]{0,2}$'))
    .cache())

print('Після видалення дублікатів:', typed_df.count())
typed_df.groupBy('is_product', 'is_cancelled').count().orderBy('is_product', 'is_cancelled').show()

Скасування не можна просто відкинути: оригінальний чек при цьому лишається, і його виручка
рахується як справжня. Тому кожен рядок скасування зіставляється з рядком продажу того самого клієнта,
товару й кількості. Віконна функція `row_number()` нумерує такі рядки за часом, щоб одне скасування
прибирало лише один продаж.

In [ ]:
goods_df = typed_df.filter(F.col('is_product') & (F.col('unit_price') > 0))

key = ['customer_id', 'stock_code', 'qty']
w_pair = Window.partitionBy(*key).orderBy('invoice_date')

sale_lines = (goods_df.filter(~F.col('is_cancelled') & (F.col('quantity') > 0))
              .withColumn('qty', F.col('quantity'))
              .withColumn('n', F.row_number().over(w_pair)))
cancel_lines = (goods_df.filter(F.col('is_cancelled'))
                .withColumn('qty', -F.col('quantity'))
                .withColumn('n', F.row_number().over(w_pair)))

silver_df = (sale_lines
             .join(cancel_lines.select(*key, 'n'), on=key + ['n'], how='left_anti')   # прибрати скасовані продажі
             .withColumn('revenue', F.round(F.col('quantity') * F.col('unit_price'), 2))
             .select('invoice', 'stock_code', 'description', 'quantity', 'invoice_date',
                     'unit_price', 'revenue', 'customer_id', 'country')
             .cache())

n_sales, n_silver = sale_lines.count(), silver_df.count()
print('Рядків продажу:              ', n_sales)
print('Рядків скасування:           ', cancel_lines.count())
print('Прибрано скасованих продажів:', n_sales - n_silver)
print('Рядків у Silver:             ', n_silver)
print('Фіктивні замовлення 581483 і 541431 у Silver:',
      silver_df.filter(F.col('invoice').isin('581483', '541431')).count(), 'рядків')
silver_df.show(5, truncate=False)

In [ ]:
write_layer(silver_df, 'silver_sales')

## 5. Load: виміри для шару Gold

Виміри товарів і клієнтів будуються з очищених продажів. Один товарний код у даних може мати кілька
варіантів назви, тому для виміру береться найчастіша назва (віконна функція `row_number()` у межах коду).

In [ ]:
w_desc = Window.partitionBy('stock_code').orderBy(F.desc('lines'), 'description')

dim_product = (silver_df.filter(F.col('description').isNotNull())
               .groupBy('stock_code', 'description').agg(F.count('*').alias('lines'))
               .withColumn('rank', F.row_number().over(w_desc))
               .filter(F.col('rank') == 1)
               .select('stock_code', 'description')
               .cache())

multi_names = (silver_df.select('stock_code', 'description').distinct()
               .groupBy('stock_code').count().filter(F.col('count') > 1).count())
print('Товарів у вимірі:', dim_product.count())
print('Кодів із кількома варіантами назви:', multi_names)
dim_product.show(5, truncate=False)

In [ ]:
# Клієнти з обліковим записом; країна клієнта - та, з якої він замовляв найчастіше
w_country = Window.partitionBy('customer_id').orderBy(F.desc('lines'), 'country')
registered = silver_df.filter(F.col('customer_id') > 0)

home_country = (registered.groupBy('customer_id', 'country').agg(F.count('*').alias('lines'))
                .withColumn('rank', F.row_number().over(w_country))
                .filter(F.col('rank') == 1)
                .select('customer_id', F.col('country').alias('home_country')))

dim_customer = (registered.groupBy('customer_id')
                .agg(F.min('invoice_date').alias('first_purchase'),
                     F.max('invoice_date').alias('last_purchase'),
                     F.countDistinct('invoice').alias('orders'),
                     F.round(F.sum('revenue'), 2).alias('revenue'))
                .join(home_country, 'customer_id')
                .cache())

print('Клієнтів у вимірі:', dim_customer.count())
dim_customer.orderBy(F.desc('revenue')).show(5, truncate=False)

In [ ]:
write_layer(dim_product, 'gold_dim_product')
write_layer(dim_customer, 'gold_dim_customer')